# 🎬 Montage automatique Locapay

Chargez une **vidéo brute face caméra** : le notebook la **transcrit**, **coupe les blancs et les hésitations**, ajoute les **sous-titres animés mot par mot** (italique fin + mot clé en gras capitales), les **recadrages**, les **écrans de couleur Locapay**, le **logo**, des **effets sonores légers** et, si vous voulez, une **musique de fond** qui baisse quand on parle. Vous récupérez un MP4 vertical 1080×1920 prêt à publier.

**Mode d'emploi**
1. Vérifiez que le GPU est activé : menu **Exécution › Modifier le type d'exécution › T4 GPU** (gratuit).
2. Menu **Exécution › Tout exécuter**.
3. À l'étape 3, cliquez sur **Choisir des fichiers** et sélectionnez votre vidéo (et votre musique si vous l'avez cochée).
4. Le téléchargement de la vidéo montée démarre tout seul à la fin.

💡 Si un mot est mal transcrit (ex. « Locapé »), corrigez-le à l'étape 5 puis relancez seulement les étapes 5 et 6 (bouton ▶ à gauche de chaque étape).

In [ ]:
#@title 1️⃣ Installation (1 à 2 minutes)
import os, subprocess, base64, gzip, sys
os.makedirs("/content/montage/polices", exist_ok=True)
subprocess.run("pip -q install -U openai-whisper", shell=True, check=True)
P = "/content/montage/polices"
for nom, url in [("serif.ttf", "https://raw.githubusercontent.com/google/fonts/main/ofl/playfairdisplay/PlayfairDisplay-Italic%5Bwght%5D.ttf"),
                 ("gras.ttf", "https://raw.githubusercontent.com/google/fonts/main/ofl/montserrat/Montserrat%5Bwght%5D.ttf")]:
    subprocess.run(["wget", "-q", "-O", f"{P}/{nom}", url])
open("/content/montage/montage_auto.py", "wb").write(gzip.decompress(base64.b64decode("H4sIAAAAAAAC/7193ZIbR5befT9FbckTXRDRINBNtkhwoDUlURS9lMRgc0YbhmBMASh0F4k/VRW6m2x3xLyBr9dXEw6HdzQbsd4Lh+2N8ZX6TfYF/Ar+vnMyq7J+utkcac2QGkBV/pw8efLkyfOXvu/vfL1eZeFx5IXbbL0Ms/iHbeT99A9emr1dRN7z9TTchG+9n/68s3ebfzu/jWdXP669SbLNIm8eTiNvGi6vfkxCz9v71POyJFyl0yTeZPF65QXfncTpJkpa+hI1p+vtJvJmUepNFuFqmu7webrepntZnCV4HK5iNJd6y3XmbcJEPoM4CxcC9zxeeXfk2XRx9aMXrbzjJEwBwoZForQl7SXRNJwlGDOaW2RRsmKDdzzUOI4S7916vfSmQEq82ipcAItgAyw83y6ibcKvyzBBl9LgYn281qLRfB5lKSBerQmtNpnqu+U2FSBRd75ezbxgLVhYRYtF1PZm2+mbeHWsEH794p53GiVZPA0XXq/7oHvee7jf9TbJ1Z8y7+oP3mY7WcSANbhTwWgnO8+8KPM6aZK1Ojs+5jdebtZJ5r0GTG0AnZ20vXXaBhLaXnqyzeIFPreTTbKeRimen4Wn0c48AQ5mYRZOF2GaYiCmkfxRG6iOFrMd2/pqu9y89YDq1UYrv3j23FZ6tgSq2/rxRRKema9fxsS9/QF87+x81/a+antfvjjyBjLqtsdht72D7s7RSzy796Db7e7s7HzkDX7GPy/BrCw4/Tv/Nh/Qjvz1Xkb6pr/j4Z+Z7XGvj+WQAAD/o/0n+5/f7/me8+8j0Gq0BQriLF8vgaWZO6TJVqm1/aK17uPP9x/fq7TGibcNScXNOo05u2MuhLEuhL43X6zDDK10O4eHWrGLXydYxkAbvk3CVGqvJuNIYBkbAPpevGLFfXmtFDs2FJuDppTrO2Dlj9Zbzw+30+3KlwZO14vtMhob6nbh6j2QAul2s0niZZSMTyIMJORQ0NFkvV6g2KsES0I6iLYgzZPtsg3QTzqdjo493KZoPDx3G7533yv/+wjca28WLa7+0PbAtYR3eFGaEecbMIIg1SnAmj2OxuFpuMpKgHbdt5sKenv35e1sm0TReIOmy+Dsd+5dA8l2VbAagSZepWCFVz9Ke+QzRSs9200ERgWmgColFMk78plxtMJExqv6WwVwzldFqzoF83h6An4xZgP5FPuGuBbxNBqnURLPm1+RhaZR5d1HHrg9GNJ6Hi/I0WS7kDdZGIOhjbPoPItcSLqVGVtuF1m8QQ9hZjjqIjSVZQNwuL7OzhpkCpYPeLhJWCq+V6WE1Xo5Sciez4VLo5YXzokAcvlolUbLiYEU78bTRVQjxWqLmEd3S3n68vGR9/njF89ePX7+5Eh2IQVJV3kJWVhUS2w8fr1RKScDxYrlXiGQ9j1Tw7uLEbzeplMs2ZQ/sBvpD+mF26AspLynzXrT0Eu4AXjCPKQr0wfKokluQlt8ylpW4GUWxy4Jmtan4Cszv8BxmIHYFqgsL/gJBg46yYAitJysYz7Ehp6ypKF4YUNKoFXaVXLC3G/RoXIIL/hsL1kvFi0APLzwZ9HEb3s+avtekEXLjVBIMo1a8lTomwWyt5vIB3FwV/mPp/EsWl+OpItS431vEackH9nGglk0D0GOYwgs2Tp5O+BL5RgzLGDuCE38resgGrSb7+9XP4KTgASxUU+jDHJQGgE9IEage7kGyOEWm9gTkM+333zz+OkTbncX0ptvEIsB+NEPA4vUAVbPQV9QKeDLb7+tVXRq6hUOKxUe9I/D5TIcdDsPH7Sn68U6mYTkk9EgSfEQHUzSwZ58SZZ8sN+fLOXBft5VPsn1/nr3y/319ov+Dm0DQhgNsFbqdmvw7VkA8XmPcLlgFWTW0PQDt+lu5xPWugT6MeXeSXS+nxxPgpOW7vcn3EA7IJJ4E7Q6C/3if+QrKSQR2ll52XaziAIwn+BkGPdj7PD7I+y4hy0sqMSLwZa8ADLLftu712r9fGGlJN8J1GPhhLNos42xb0fHy2iVpYH9YoYiK33gDZX2CRkKEDZbTovZl2fmVec4ygL/bJ3MUiym4ajl4eVwVBQWDo+GgZvgbCgl/VHL4qxULJ57WbmiAEZi988wU5BT/NQ3C4qNpVmYZGgNzyP3ebRiH5e1ptDBmQKMfWgSTuJFnL31W14MMRT8+pv1Kqr3LzAM/Y0/AiBg26tZkHfkNkMwDlp18IHXDrgqYAqWJbrgG0NX+EpBrzR1gZkYiOTLxbkXfB1OvccbkJJ3hA5BsehvDpqNkr0zPRVpmWerTBjti89RYI1+w7go8Pl6EU5aHd9sykbg1o9FPOmoeA+ZJcMkL3cM0uzvTvoW/S2DFijU878IkzMy1xBHk7zEMgRjXUWmSJgsD+9piXIfHbDl2RgwTQOObmwA9FsF/g2W+Nq3gFzfiGLipna0xC2auqENRae2kYQx9uOXW5z7ltGTJFkngf+YO6OZTQoT5bMWpDlslBQKAgzKTkptFiEul6cN80W+8PW3XzyB/DD++vnfypIASJM1GRgbm66Xy+0KdHjXVNpbUDjdOz3Y03JcO5ATFtdXkNd7RHe7SsX+MprF2+X1dfW9Vobgb7q+BXCscZkP7cvHR6+evCyP7oZx6Jd2CTz9VgEi/24ZuZ2YJApky8dBdw0hPBr4thfsJMdb/J5TSDjF4WqyXYQoP/D9YmG+Ks2vUU54OBBPvZN1suZh8Tgqqx4McUxP1nEag4NCcBJZQNdNsTRNucE1vMGSsS020IVSkKxZ2s7iyl/hcIXNaF6BnjTZ9i4UD5etf/n9f8PA5xCBTgaUugrOJkAVzXYsZJMclxsoDcZrHN/m4yTarAcO5Qr71T4szlsG2UDVQLHebmTClX/cScZce1hUEO4EyDbWGITXcDEGZ15usoEzcdyYyOFv1TjFLz1C4z+c705jOUnjiDL4MlykLjaUNTTvsQn2KPMd+0PDnBmm5Oyt1IWUmZmdSkNeXxNp75vLMku51bRyn3W7CMqLsnniZsAMBC6fukC0i4W+gR5vTJl64AOyB37RATABNdCY1NNAM1US+Neb319obkvd4LyhMlDffsW3sw6/kGGJnMIH8o3iykoL4BO/XDGCT53fl9eOJpfCglT6SVXyao0uVXozAlpq6U6JCKeV6Yn7wGUOmE05u21noS/SGAt3+LODkUMLEkN6WHBzj4AnFNxs8/1U6g70mWz4SiMCYM5UlRGXuHPBvZ39Vqna/5f//If/+7//k+e9CEWX+vTFb3C0Q7tXf0TDu8L9dz1u33HKTXUDqoY2ZyUKzSVUHikVoD9O47TV8fwKIv3H0yw+jd7xsK4NPzlH2a2sn089LIF4To0peiI1e7PdyH3/6h5rdRzyNuMd2G1px12en59wvCQcPdPpEOyGAe25XaBQnEP/lXgXwOclF6vRQ6Fdy28hfM7GUjxfj2YZ4sNImD9rkTWT3C1Wnjff9A4HQU4KpKPWNc29bxW2qdacrNPIXZQfwmx3dr56cgSVy6tn334jB2Yf+kKS20m05Qd+6U8VF06W8rFcykP9C7WifJiK0Qnkh6NX377gcYanCN9fiBKKGpcFCQlqEVGb6NF9RjqENLeEpOjN1tAwAlGrGJqcxHsNstpCxeBRme+BpFYYPuQM/IkXqTxNd5YoRGUAiRmLMfYWW/zPvYN/Uu8ttUu0mKAfVRxAJe9d/X2ILryp9wNKU7/wwxb/oTZAADxX/0yYQiobQnxuz3eoaUz5LgU+PdGkLZds6E8ZtU0x1fThKTUsIRcMS2KcGzyVFZclV3/k34h/oCjaQMsi8o7IQSmV2qRoqud2Zvw1PUErpzRzLDHwDP+jd2AJyjwU99g1O6YZAccy0Nep/F2tiSA8j7nPADdQOiXUhqV4MokjNrXNdjLikN8i/Zvi1A4mtAYyVujmdB1D1yofmDh8w0c8jTlY9IFhxgn2X2jm50La013Bzi5m6oet+cvJ2Xm9C8Ss5O1sF1PPPyCGXXSBsS9jTI02CNUxtGqCI9EUe0q3Hk+EfBetplCkUwn0GtDMotehd4qzBel4B/o8DGN59SeMdgn2gd7JWmX4qEpMy8xuom3GP+dAq/4BVDH/TyJQaSeF2jQLoF348vGzz56L/uiDCZj0YiiGZDez89ow3QXh7aC9FYn99fvJvU7J0w8ikB1LIG+5ZEBcwBnXT+ZSqFBsTkHuTOVUWUKYHheoYw0ys0EZJpREHZjBgsQf/ofvz67+cPVfrv7r1Y9Xf7z609U/XP3j1X+/+h9X//Pqn6/+fPV/rv5+919+/3d7I7IQ/J+BgZ9FCRrvQDhewOKJje73f8e3u77tEAoDMVAmUN1DMzgVHSTPDimFrPxJcRJxHmIr213QGnT148BYhR4ZHSPGuE0Gzvd01543Mu7sZJOXuQJoIysg1qEKQjDYR9+vRn4JCHI13929IQ34A58VpYWySiWEtQbdyBvTKgpDH1bTBYVWSVTXyQiwQ5mVvFSL+plJSbEUz0WvI6Vrh3lRv/A3mERZ8bUk6Hxd1HmDEtIdNEFnVpQ3Xbxh8Uofag1ZTdkyxkntAmQC4q/d+au/ftTHzj66828w7Gp7drOfxVOI29igBzrWNyPoDQNpctgdiVJGmldBzG8VLWA4zfomMkelLp7KIIfGSSBGF8oPExpoYe+VVqlyDxcNR12qeCLyCalHNiBqQ3PWpb0E8pY0rbxhaBseQUrVo67sXmtKH9ukOO2CHZxG5CwDbdouQHcabZnaTJqpszgx0IsQynqks4j0XkY/sW+bHO71Rg6ZuY+9O4NSs8bQFc9EWwppMzhrUaVbyOO2tkKVraHzaVN3qXSWbpeBVG8ZtDuEB0v3Rpp4BxK2zbS1Nwe8GZoJ6LmwxxZa3sesdVd7aiID1Z6eGe2pqjAzKiuN3tQ8oSNEeB5AUX4foKFVKOL3W6bcBuWgHL8s6CwjZmbXEJhgDJhRoDfvWwVnJULdWAq31G1aJc+JYVG+iQ8+NoxTdjJaZSL1wnC4oxSFKRKnn6sfxXVluZbtFKI3kAFLEZmy7IlC3auf/glaemx/r72Ara6n0y2ag6HBHi+wglSfE+m2yFKbExgUopb3qN6dCOPsDVsrfkJKeb2mOFfpHwcOygspDxy/EznidwSAtbiLxSICEsIW9zHKOmoB/80qb0SGxe5ym3oxrIAbL6y4FOfU6k1ZijsulVYlRZSQunKkVplDKu1OY7CoSEpN5f2U7ytbxFDmdGr078BGFPmqJVb0+8U6kOopi8wC03bbexPBzBYuJzOItn1vj+eq6dCPwTzdraeNWRp4zsu2/fG6wrbJGoKu92uU5p/X3q95dhTaalW2HOPcExWqjE24UtVcCpPOa6yc3mjH1fjnT1Gq4LZm8CIGijQA6NgU6B1nF39kfoHrwIbBn2YdFU+5V7TKeKqhubQR34Dv94xREM8pzaSvTE7ysgeeKxGcyyxlxbBIojqsVi5AiRqhwA2xRixL47XpWL0HpDVdVd7IZt32uvljHIwB6Rt3AvtVYeINZmPFec6LyP6gMoTMGEucGxQX48OudxwFKwoXA0VJXRpxqeFNX3oa1QqR+9/51yIG998bdrMqPSYb7TdBlAsKAvqo9Uhr90q0jKGRszdbrmS3riinnRmwhhPP77yGQsaIT6IE1eGnpBhRRBpmP+Sm4Je5zC9gFT2Fw98szP0acHhNxaHr6Mlvnj0ff/7tN18+e/zN50/UPcruYq+jjGK3jLE8oJsXRUmE0SUg1JYJlaOAo5kYWYkfwsXbNN/c0mgbLwYV6Ipt7gvr+UgXmJLqFzKZbj0pD29pfCyy0FRPWDz4wqFINVv5Rqi4AFxU3yYQ3QKsvHlMU3quosJpmOygT0fNGKtQfFCMkcQcphbipojpcza9hDYyyoRXP8ruF9DUIcdZBVK8nBLo/7BP62HXHJX7PCRC1QRXNtnBj8NkRm1ciB/JKr76Y+LsrXNKjjOVPL0AXyY8/+Z9hBMgH/4p0BHKyRtLMKFO0O5wBl/YxMXgQubiCGXYUoQQI7gsRnALiKochlwWTnJl2T8RVaqsK7zkeh1VWRIXA9a5oCk6lUUOxSLIj2y8JLRCWLISqxatHRnyNZ3wRBFJq3nn3ZFyFLTe2284TZnxW2aAKmAEFhfFfrBNbIk4l8tLA29oZ6f5hPUGu/LsvIxVU9uB70wkCt1OR4WzBGqOHE3xeiLFztREn0vhqOyY+8tGfmdQIHQrlLOlFuRo7hDmB4VQ6UB4E6sWhhpZEKmp3NMDs5x5pYKIr7/GbB7cv7HNYhfDMk2yKC0jnSgmiA7wwklkhaJ9YROcdQvOpwPts3Lo1rbtzPjOUpf1LBpBd0X/dctvNR8lVjgIcIvrYcuiixUOFmmxZYm7lTwq9i0fE0YHq9l5syrYF3ZJM63dJc50l3DmkkcQDhml+IFfZkh4YL5d07gyDr8v+LNFaZIpP7o0pPqR8qzUYVr0pnM5DJkT0RfSAwBcSvRexP1BznmhSKUwZHzv1Dk6ns/hbWDNLkcRDilgsF+HGRQxibsywvK6APad2YR2cJBvTOHQ4K4s2ZLYXoctkMf+ewQq9jjxRDuaiQDUE/Hl3qgmP02GMtvkK+iVs0x2sn+/zk3gSxm+KT19PXFgnjTALEJUTLVHBS2BmhVeQ2n0Gm4r4pRVcRoCVNtsrCzdCJcydvN1onLe68mwb16JJPc6rHFjAiCr51Bq2Ibw6ECcmtyO6qMGUixFjuximftaOpJjHfWx+ebsXUyGu6vd0aVvZV+3+dtYqJV9zP3S3tc3e1tTR60mmM3q4DFFzCw3zGbpkP/5t3RFhHH4t4+fP/tCJBkyLv+bn/6JsuwXsgXz25egFHy8En0R7VWQPajQavHp51bM4I+r3xeyCX8/VdDw7ahw7PWtuLSAYBOl40Ksq+wfVk4bboZgWeBCquTYGDKmvtF5JvSszzY5fVYYii9ThdfCh0ZkycKOzTxcaGO9bpf6mOEuS+3CCc37VdWdJrdr+lhqvvfT/7Jsb+PQkIqN7hOrDBEQ7bzJD/Cy0cjZgBQRFlNjWhMgnpbwwmPbeavwXFSNtBVP6U6Fzoj7Lv/AbqIfRoXNB1GDrtqZDZVic8lKp6tBTSN6f+hOxPd6savGXHXw7heBAG0r/6k1qaAHatWMMlK1Q8ZzOIjOOxSMESQjUvEcMjLsYlyLGRXjkZottqtc+sWKmTPgxSowI1FgUt6kn6iYQ0zRQnCsyTWbEzNUq8SrIMCY6Av2oYWhzqOqKpBf1CYORUoBIT1oDfsPCnHAqGsVGTgkWMdKqTg8GBlVfNtMuj4+HFUefFJXhGiL79ktTNm04qRZ0zQKiXBn3pzky00UjfxtlppuwMJzLlvv7bZ8EiMDt0/YpN1P6mDxBcBS1ZXKkY7XLetSNhm9B4AbldM3Kqj/f6DnA3TRrj66oorOYSl7+d5OIZ0rpfMR8FBjW7xGQf0XK6mtapovK8hxdNIf5cKYja/hEhZjCrScQUpfDgQ6XP3IcAGSv0h6mZ6TwZkWNnJC17XDWAXR5fXsyEsfTHalffVnajkYbKRBLKLENVpzuJ1CyE80FslwZon9aXsv+3kEW8UPvFHdK4obe1LNj6+9aO8gN8G87DSHb1VVSsNlvWlRydYVQ2rIKYxvZtvy93iqKylTWiPXYFQ24am/7m/DxbbkrStG4PVCDPAZ9KjC67nLgPWfSmyqdfK5xjRo8DEQ48nSnHkMbu4w6ONBfrRgyMq79Urt66bbYL3yjConD2IFcvMwtpaJ/VpFFff82+gltJp4Y9nJkkd6OsNKpScNHkNaftkpIudKa6qxwsAM+5H7mv0X0m9cLM26/lFqFUsffEIa7hs49SBZPNJzpLQPB7t4dFnMyDthT2yu6OEdxKZ8SjADbT6xfRAJLztOYF/LrTfRepDGzAp55w77TlGTalsDxbYyM00gYTJYjlNhoAPO8UQxN7G0UjmwlQrocNxHbQMx1DX5Y50E8sF39scN84BqdhbelSkNb4oBgdbelWlNirXKMSHC3fJeyx0ZNjj0WVEG8y52lwVZE7QRTZHWbbM+yFfgPE/bhTgmwItD+J04HdGkbKJm307VbsGwIMwXwybbtF3es78U1YZlacfvnT8eu3XiHonqqgm9cn4vjAa9thwiUbpqT1qKw1tbDP2KmNk5jt/gpCPyjvxJXWeYL+MQPOJlpxxiKvviJ/ctkelpfWlVf3hVP7ZORYINFCBlZHdMFaqt9uu6QqLLkswwpM40I/0RP1NjJWgspwCZPS9H+gUw2qdPiE9NEqzhPieIWh7O4jD2fqXmIv6ynq0gxoBeJK2K/pBtmg3AhNsV+7fsuyzgaCuH/loIMXukcsPGomwj82xCsJMNAc1cWtXgTWHi1ZhT8+pnqJJl46pugX0bNWhPKyqw1I5DVXKp6aUdGN+j7r1OFP9wVfcKAYeJETet9W342rW0vRY7JuBgUoVoFTQ5xdygMD/o1jTmZTeP9yvNhcgIwKfKZXvkFZUAYtEC6WA+9Q6x2K4teINq3rC/vxoYoPT3X6bL7zxo/aWz+35lfkM7ZgkY30/sirGcoLWkHJd5no5AihrqAt8GyjQ//blNN4cos9/QwE9/FjtRny6BGwlshlCeNyTK03zVHMcFT+U8Gcha5JcOAo7hzkO+bF4Pj2MsvuJH2UavJmNtrkVc5jq/1TkVp7U5pfdVT+Xb6htqCI0yvzbzhejKCTyWxWEIH2My3pjNlW50XKo119RGqZDdd81KXZ0rKTXTX17bFksLKrQNsjUxcHQrVLg678D1HlorRrUedxA7Lo6UfPW0vNEexw12IDu9H2YHescdWURRY47QgRUFIP7Rhz63VRSn+/PMsq8KtRRDx7zrM+NjkJOgTHsveljIAPHKyI9ot+0aQkTqOMxFtmIpAtkyQKFGIZYzY9uHs0PLWgXOylg6Sxu4TGw8VU19q84Tl/Wcvt0OSLv3HK4gghPZmgDVGvZk6PpLWhDqZw6EVoNJ7Gkh0BtR/SzNrUTi62bMQ+Jp6KMVbvSMX/Al0h+/2J41xahLVWKSkQRPwWnNG3NkzrvRI61MOX/KJ33WuOFTr8JP/DazhidPxZyErV3sSfgsq2h90VTYd1z6VtTi9AoTyFMTmJXf6bZsbGED3NC3MAHBgMMrlKAvcj+zhoRBqUSQprmwgecJR0kXLehIkT9GE2DoGZUGwa0GBVi5GeSp/mC/k77B537HCICV1LIZYqgTMWysfOj1jBNlQM9uPICZgeYF8cPW7ulv72ZVyLWi+nRgPkG9gXHbOu57wbG1I93xjo2uSARNpeaVPbOp19bLTi0XTavkEhqscrp+KsR82LrOOzRDm1nPghU8xeJutfMf9P/csauRizEu+IzuPPtt280eoCUAklYGp7incq6xai/9pWP8NZO9dB0fOaJltekswB83dPwm92yTLaD1lTS6ks1IaGrIDkY2jKhyRlHXL2mzGHE25fi6wO0U4kmA4e7hZ8vRNk8SVSM4w5MR090anpnhJA1iyle6Jzo27diK1g72tbky8y9b/1gv9zGKqTWbJBVfvhji/yQN7ERQUwagp61WKQeSAG2zL7zs5HmWMIX1p/utsqdBmXMav0IZkkMtOm/Kh3gqsB3jXPYrJGwoGU+cFsyaT5PpPjR4cggBVZl2S0ePoeFHZf88PWuoEu2QBJNhi9GTSBmxuW+snFqQxqt8UDGvhR///BQSVGBaR16m9AlgjF2SaWu6HcRmIUmMKPaYFwzacDPgLHnrqASAxTxXF4LRthED6SpNOf7Ebl2pD+knG+c9jSdvx6twiVg2+8RRMJ1Po02GCD5+MMlN+YTCdF0VPM53bq5nijXAL+MtwNf5l+wxVBJZj/q5wYi6fica9DDnuW4yWZ8bQ4ZJWEWGc6CHVei/mVMkweQuRJP/Md+z9h4PqfaBWp8kbZHBcWcVnQX+c4iGAdvAujDrPs+i1uGfQCq1Ouw/CNj1HkHUL+zgnjBFHQKGPZjTfX+xGPTum/Zsr/IJl27uNoGTn63zVAKywtVni20SPLT8elkG9OXTzx47sJLhm/9aOK7Eyw7CiMPF5iQ0ACt1iaR2y1aaBo8Grhv6NcMO9u/DGb70Jx9RRwAc0xWEu2kkzbuLMV62dfYS7UomEQSjmeNeyMoy7FPW2pghluMxIyXmFHYcj/+ITkNYN8D1x9OTViNrWMnAXAfsE3KYddphPD6MEjAxpsGUYuX0hNufQ+wg6TF5rO3Ev3skqT7uPgfHDpO3d7kG0rtHVMGJV3m4uPs0WifHceg9YzLFaSfL5v5Nsdf+3W2a3E1xfI7uzqU1u6TuMujtdHv3C3z8dnvExGJ7TqNVOCcfCOdjsIuF99l6MfvlYAQj38tbdEPcF/PORtw3+WViviT2SULYoWl3Uqi1i9+aN61tp8N+mZSbnyJThAnUykmH6iKlGknIlXOncgBTUH5ZjmEyZ4Wii5oLkCb4pAP5xG9wgs4rMlZpkO8cOSbs1uE/OYdhnsjzc6y0Ku7Kpd6mH9xb6vT2tQSXGyLNe0xbt3CQvn0n17duRQa3KeuYIRndaOR8WXbOyDogX/HHkKOGJn4Tx/08wZuvImIeS1grmWd/syVtpiEYwZgRcAPGaI4mOirHOeMFHB72kMWM9UuHBXMm4CnlJKT3hqh5JB0cVJE4IGh4qkl/J6nkwqt/zIqjwZtxlltl7rVNwqaXHTcDIA8MUNY7UQ6SheqpIzVhvdfdJI8L4c145pQnlEdRLoJjq7Zv8yvPoKN6rEBjC7m/hnQ+DMz86Vmakyi2a5zUjfrccV+saESbyU1DhakI+ICWh/2qlaA4yA+DIka13XnU11gruh+c0dvfGFMNrYkKPn/DburNavzyh0KofoUNUBbIXEiFhfgeCRraqhdQK5v4NJar80yW1hXW0or1v9Hm63gWx32luHwNj8ULMgiYQvYBZaDg8EHb+4Thb8F9fDuUb/fw7f5+q9WcJKyhTY1Esi8gu2EBmIikvIw+bGwxVjoPHOkyX65iIkgzdwBzcQgS/mwWvemYW36Wtm5yKxSkcPtgyRx7zYrpKN2YkXU7+w8wgLyfxuKLM6P8j6l723c1eMuUKn+297EqY+XJXjUY2PVtPeMh6TsxdT047F87pLoXak429kDKznCoP2sLQxLgDirAtQlbJcYGFspE0rjeK7uenJhRJnaIiVUe6CClmhmmeVob6FuD169QkGJBPY2vnI5PvLtV25yAbcezOBHA8/7raKKiPPiOYik8hu42GfrYYJBLsGcUsU9aljjQUzPqhTc7CG6jpz1t4i31aIsT/ERD9iFj7RobOqdF7uxMqaNW4i3foi2D1hsM3NbBbpjzqhqLyneEOi8x2WXJGh5CdfsAappPwCEOu40x6VLWXfLXre36unZlgfIS9ylWGIlNl7Fdqzeu09IaPczXaB2WX2Z13nJl1lfl4sQIBI3rr1T0vC3ro0S2yP8tn3x0zar4MCL+RQi4mXhBZ9K6qJfkmxUEI7iV5R6658Z+QKSct0WzLO4LpaMlNXsB/8Ba9fHH3kEeIY05TsZM9htRa38agiPIIAA+sNAu8gAPJAGw6VOjVWAzrypA7Nu8mqZ1FMcpRyzXrmoHYvYsiU/Pzab3tsXjvdM014sOf4MX6VTs9g/2xVbS49YS1XuXxMVqeOl01fOA+BJ3D5Jqr3OYK4nRIm1UDx8+dKL7QQYrS3Y9BQyn+bN4Bsb6MapwnZXfIYPM8UlmXjop9vY5k0vk8Ujjd1EQaMstk6K+89mz58++efL4ZVFDCKPobA+wKB0/Upbm9LXHlpS27wgYvXtchpzzyMBQ5nYWmnzKMPJe/fU+1KSbt4FoWfYLNQt/gN1Aruf9AvCN81vYf9iv0dOeSv7s4BRAhLly9rpp32+Y95+vlqzclRCkb1cZknRnzP4FXjFOVseqX08Y77fs2CTNeB58YjV149mk6gzf63INBee0Q3TzcmKWHjN0MApM0OMcFDYHVfyAPME2kM5GNEsROmJNFYbjaL1UJb+ttcJb9RjH63cIvE8Do8LEIU7y74vpoUuinpkM6He9H3ZK7lU2HrmsZt0XbwQoeVJhHPiyifFoPqXB9K535JDgCaiLnRFcvt2T3vfEq5YQ5AUFHNDkHC9Y6ZEUNA/4UkbDJgZFxVomiHTORJHrdXpChzrkU4YSbRNP8YUC9rw72L8v6Bncx00NwFFMF+PBXu/AQS8pCJXRLQbyyMuqRpS2J9jVNbE61ddnCB/ChkptejwFOWbAA74Ju9zvkEBlLWW0Psk3vpS3Of9I5czvkAEpiSn8kMYCadM08XogeXhlDHR0Rv8lUwFbBzQ0ccAeR67CPRXK/ocsTWLUIbcELXiEgTqoo/E8x8m9Ck7AInsVpISWPmTWd3L6OKRR6KHEpqAUEksEe0QIaK33oBgsXpGA9rWUkBA+p9sl5YO5ttmqN8EQR8OaKm+6PasudfBRRUcFDfTbKhAA8Q5yS5EN2Y4dk/T+odvozIZxkYiz6lgCMemZofAeBwpwYF18JafYVk26gH1pXrYvDXsHvQck897DT+7zc//w4BP4qj+837Ph+jgJDrwmdB3cEll7vYclTB1HK1B7MgbGgugUPyQLXts6+6vJpYLBQF7SptCymHxe5k2PvJdJA7dCZ6nm5JZ1DTN5scjbuj33LfHSzm5m0Dy0Py9drzh4JSxpNitjshiJw+4Q0Y6gnzmpmoAggcNy5IRipPnogF4UM2NzNUUpRYJ+JfiQOwf+DvfSPjw82UoRmRtZjw6cj42HGoo6goDmfwhcprQnnE7+RIAkNSuEiadpT1T5Za+nYV4Yg0g2BqH6Mke71s87ez5M+5E4+BLevrwdKSGnP9DpppOTMOBqySzetsodrWKudtC5ny4Y+7Eh35u+CYbPcURJJFZu2JeTjlCRIBlP93p5Jq+zGBIOLwfqMJt1bvbzzyZw24f/2Jnj4ENL38rIHmmwD5jlUYqclCIpFY8QyGWow51VZFpFLEckL9MAahEAdLD/yeEnrQ4S4dJqhwEAWiTd72TryVsE1AW/hESCLECzrYoLRgeiN4dEJuOmoVub+clemtRJtqsALgNzBoJLmN3eqSSYZFyC/IS+GLEIYwQ6ROFS0oOc9rv65mR9NmbGuphpQ26yZWjlgSCwrVJlX5+NUwA3ZnbqQbLWAA37BmEjqT6Mbm59bz0nPLwligmyOVrMP7LN8T6KMUYM0dLk7JWknuLHCiKa2VwiFHFYW/KZphLmPPTtgOl75RpEZzi8cxw8wOO7jsaavtdZvlwl8+2MbGRmfGjtSMeMsGu+ooD56iweJG1dOqsY2KQHyeQqGusUIOQVLFPXQvxLnkaRRSEgRtHvxWVLf1vkwkjJkt38rEI+j5ct8Ifegy4ZwkOHTRlEgMOeuXsEH5tNQD2UZCZK1Gct3gr6hxBhidTgvEWBJxvMzOUY/i0yaVsymaang82g+8GUkif3s7Gm49mWV0/94ssrbByxWULIWxad32o5/NxxuhPGK29kWfwCcWGw6JiYME2TrMnD1NULe4u1L2uRwNw5A6UBXV1imiSpEVlU/Ncep3o3kldrkc5hDK1Cegqq8eUaOhNyu8ij09wLBDs7JvhMO/b6Re62fOfQCHm5SQ4q0ZZxWNtFBmPJALpdbL1H+k7gJHvZpkjXqjle8ULd4T63pivpyiSgYdSJXtZUv0WP18l58V3acJmJNJMbuda4W85789U75g1YSC64EEncMhN+LPliV7u44E5cpSVQgOcEST+jGdjMQCXvLoxymO2t3NQ3yfO45Zd02Gg2rXKLiDbjwseMrWaKEhvENlvDOSMSd3yZWoagd2zTJqo5haz5BmbTJA1McQgx9B4Yr984SeM1TLoeleZOmExWqxrzoL05Qp0KEYMSCzNCa8WBN4/oUnbGh8JEWxUPXG2RUVfU9EF/q5kkEByt0hXlOW2CABpJuSQm5gkWRYNUCcyqoX7uf20I6kIchS+94MJ6XuBivUg8l8yQLun4riSJ+YC3vMk9rJmPNBQRpO4mF9+ozcG2KOkC8rlB8oG43z2YXXaWm3tOpelSXBnBCJeb6FgYHO5zW0CsXpTZ4VtlfmR5aAtD7ncO5pfyMNNnOTKKN7FvDxaF9pkLZFAw6eoU2gDyqoMWAb1DSPfmmhv+dB7fove5H662iwV87wbJ4OLo5WV/uhjw3oNo7Rh1i8ZP51IpnTK36cV3l/2Lry77c04bLquIj2ncHoe8AwYqI+5w4PvThMrJ9hQhZrZGew5H3gvcOXnZFhE1GfTaZmd8uz29t9/dNOwU/l6oncOzhFItAQDA7dDUNLLveBG+xfpP7SiaGlqGG0nM0LcSYf4gxAM54KyNV7Lfk2dNrUz7siXCIfN8H1f0sB3aWiNJ2gEe+JY3SMjjaSKT0ju8ph3ZOcNwKoUn+hNs8438DklgzFBwJCZibrDMZsGWSdIO5ZR3b0yaUNf0TeWWCsN0rIaeP0v0pRtVdbHyGbT5coTnYV7DkAwbkqdGESebznXrzJfXvDHVd444crqRNzzcYGBM6D3DzawDf5vN9x7oaWdeDpVUr1AdS9XrUY4ySMHBCJVd5gjZFImidyVN9Pff7yJVNFn37vcrA0uD+HOrVa9iCzILhjrdaTgvEn/IMtOxNWdX2ptq9Y02tlyfzhcq7/p3SEF684WVI0b1OdUbZTvJEhwwKlANoxZ0vGMBNnWKN8ywmd2qhUR7tMpc3G47Vhdj65u7tJoCm1gAij+oIHlehowtb2fxKRLWBLSqHhx2+Q8vl/WXh+ZVWn8lTZakcP/ipN/dn4GVLM1nKp/tC5ijyMRttttIrici5MaruKTFKUivOFjfgvboAdMrUeJx4Z9s4yNGjRm7yy7O1o+fSSurL6wrPPMC1LN654lzNPeU+wMWqLAhRrUaheguE+DzzeX3qwtnhkMk3dnDpcjus0mLhXa9XZtgbPdsd+SagHdpAma+nu9XXFJ5Ikxcu/i3r8a/ffbFk2+p4vFli22LzHSqn/f08yyaLPXBG30QnsaSIke9PZ85l0nmArSYiZjBxghss133zsniPkt7HSklbd5loOEiJm89JRTE1nGPDOoBGdd5l9L+6GrV6J8m8pp8UxlK5SOWLCbbfabRG+U2zMWaxsI5zC/aHJWLyZUqWsacknkLpxyIA1/kP93JnCatqxvvzElJ/kE+MzZ3qNjTa+6Y4IpmWPHy2H6bnbftoJFOMhKXZhUR9S8VWepMbxEojedelq4BJgmnrg0yF1Gph9rLUduSG1phVYSjv4Nmp0CrrKYs8DSwQ6v7ZJphXe+x5nheC7twUdqCLXDFA4+4UjdkCEuNxP0dYLdWSzG5e7mhsiUW11632XuSoJVMpFVja0tTCdbsrLorW0vqc2Q//fffHpUBfGdsZJpkgzp9TkTjv4/Uduj9DQ54n4ENM4Wfe3t7mAec5kznjAzXU4cGjg9jfsehF7/KzgmY77ddcbI1485tvNMziXBqO69yG+/0pB5m77qMsrBFHS87d2zLn/8G/4M1r88HgfYOM7/E/pwJKHfYeHmGH/t1CuPyaKae4nVJufJCyKg23bcQOGK/XVrRIiCuVD/T6JeN7HaFrP0zpXYrqzf1Y6QghAgo6xFpOD4fz5ciEyG6aF9FZMnKLAqbgYuRZy+etEpbrFggKqv/Y95T36pEIFtmhDPmm0og1VYSk9o5sHoijGwWkCZJifXdVSzR2zlDmPNCt0nRmMNRyv7s7Aw5k0TrNRrKXzfSkRtKU47bMBRdxG0U4RW1ikyiib7ndPonsyqql0DQaeRkaSEah+uLIGfqaA5aCVmY5TQMpbXRfDlsMTdvsG3To8JdMaXIWIcXmggtvfV1xeAZe2miiE9OQqZC+2Yuih7o1oQOjs29AEadWNfOGQ8y9GfCTAJjLSn5f1fEvpI3uDp4MGCyKezOvdrC5sjQ1jR0dtQ2PzXKtpxqpRQNt1NcVK+pAwb64zhRd4gcj+Lh7t5LL6GfleiWcgk30d5xeRusFKxSyCOtsDjukIkEi+M8csuhzzLQBaM+6HZLDkQHYndf5Gz/Lr/L7tBqXb/FiWPeJ/tdWdZ2N7nrNMMMEIcq+UDrUIbKYs+Byjq6GqDQ/IfBJO3OIisSyPRpcHtop9tJVrTnHXQK04akP5b8LIS6X4bVtMnwUPOz1Fool7npLt4QJ10JQ3auLF6pZKnxw5Vbnl3RAOsJyQTR9ozp4CAZNPQCCwDyQuqNXUUsoTYtrpw8JGEpqHOm+VGcrAwMxsO07LflNHJcJNntSr6i4zwhYBHe0Lr22FYPgygnvC5daM8O3UMJZHqTo05TgL7slItrprtrrgaEdJwptq2832panfA4Kg4ErZGdVdpitnKSoVZU57FtL0RD4lwyOxx+2sK8qNVe7GKKTuQqMaQTma5PeYfinz2ZRHAfSfWulJCC6FOKcdfYZ3eqUi4rUL5NT/TdETaZo6/Mli6eVdMoXgQpV08qgev4sm/c3Jz3J+X30tYpd/QGmeaIIsoRpRrqTgaTeLrF/37B9SIdlNhMJAmDBEeNnceYsSevHjMn7+OnT44K4kKX2NH9NtVIToFhvQkTJW+dmI8oPhyJ/KA0C6OlOH1oMm8ej3AmOTYpG/QO06lN4FDsr0gZCS3GmDOMZRSYKxuZEquAEcfU4GXU0lvJEnMfXZ0idOLlHOQV7TBlWLZWPoDacjEAT9TKABmUVEQfGXxyKEOFd3T97u6WMhv8hynWS9FJDsCfVp1PCgV1oYHPS9+l1KhqbyePmzuCBuFcWmSDsTVEWhkbBNH+IHH3Vqmo1V3kZrFYYZa8l1SRFSPco5ZFzbglWum2KmfvsdJBUKYe8h6Rn90OyMakaffhp0QlCPpeNT9rqVu2XhHS3UZ+zYYrIQZaE6Ny56VBmDcFS6J8k1t+WY7XErcR4l04a3J8TvhmCgb1gThtYGWPcoGpQfRWtgi2h+eNIndViHfa1WnNlpvxadkMKdQoCrYdszEiEZRoomoU/mHq7fdQ+m3cGMzK/O7yHIdJacdaNniqs6dbLJn2zcaVW/TVaH15UIU9tzUpJvVgGq+aF+BqUlZ2G58tgg5R93XJfWbluBlPnIUm7i0rZUju6lvTx1bE/ErApayd107WJCMBkjGrfKdXZGlGC+h+15lxja2T5es6QW/MSYK55MrnMgJq8mogYmNtMmWsaxnljG7KZS05isSFTPBTjf5KxNnPDbZ3xD0jzKn7PXNzY+iTlqoSK70vFnkzk+IihrLQxYsXOmKCtm111Fwt7cmaFnhKuehLJ8/SLQ+lxvtNQTwTuyESPNlBM6bVkW6vCenpmDN0jYfloOU3l0iM2g26So0naM7xYHUF2ipTHPSoI5QEDa2G+HEO4NrNvdQZSnaMcje7TVi40feqfkfprKzbrSSSrAvN75RAJdXlSDwvrRpz/77RY6patXtQB7tQTBqNZFUVqepIJ4roiEX3RROpikhGFB19p6rJR6q2zMt+JQFc90U36ZT+SpWVN2DxX01TeXtLVREK7hyjur189djjVDMlawiljZXSGLEiZuv6UNP3BF0pnWRytRB8yiULbR7QVN/4C51C9c6flx15CeXNFDYsZ3Xmp2g+KjMEye2kK+GaVdAEfAGExjty8vcPcouG7U9HBO//G+Ivb7Gkp71r1rFFh8ryNy/ma4chldsmajB/YFQdJoYw+Kr80lo25O3NO7ZuL7L9KTpcbBgBpiM7szFYmpipsuHFdUp2k0MVmj5NCgnX0ECF2PtyC0K3mopVfFR9T3c172J1efdiNbkEmBd6McmKY+KTX/ECljm2g5OKg0VuYa/RH3tHSES9W2OBp2ywMqk8KkOfLnCNGzWk+fOzkDcoPLr2mPW+PcvZdorMvKV/cC+cxWuF5jRz3NT0SC2xZGMbS/ZXNqrRMbFN1ommJzD5vi4yZ7cfO7t9cROgpCV27iTI32hblYNHzahu5J+6Wn99M/VzfHnMqo0iQJSEQ01TTT43PH6f9kgyYtTmvp5dY1Rt2t7XuqgkeEPWvz3JmWnDrIe2F3uF5bBv9JKShXchjOUuMlGNhv3DUd1ZYbp4PyYbO0sbcgp/EGYZ2lJGq1AWD+8O2pUZgKsUoS/tPJMkdgCTWAPBMOXjDx5gWZz6NsjGiehxYnnwwJC8zdSu86jJL0e5Zyb5XzgBQ66noX7z3mzuM81vrnvouyLLXN54ccXVEH5noxDey8vBxbvhbrg7Em1Fnz8m5kcb1rhsA7e2F6+O9o5ePX75Cl/afjNnhY/cPJxFgwyGvP5sQKGobZ9gghGoMLgwOSJnusMjVlC6lMIHwxAeISNHflA0OBC7BXgu9x/ZC5g4NPGreiR3M5nH2oAtzzDdufGYGqwGFyRaxd9l/3QAbMAf8Bp4lQgI8/59B+b9+yVcAAEILR+AnqCgY9DkhUy9YnKx3s4YeDh4xoDA/qsXg739/vOXjwc99Fr2Lay0eaOnIWaId9oM9oeno+Hpm9GjcuVhj5N8cwun6wXIiRGe94epRa+4qCjnqWijxBg8P89ZsrFxOXpxU9WoxjQ8ZbGWJejv9XKLsqnoZPpd1v2Ej7nP4LD+UvLRQu0qiRe0pq9B9uJH3Cl5Epu5fjTcL2h8tjT0PeM9As7MNNL4B7l71lbE3DdIvXjZ0W8W5st2ZZEcXkdzetaHkq1TkBy+D5fd8iSbicZjzj/DZgCkuScSIsggO8EHSFJXZF9N7Yd9YBVhYUgP2E8QUwEcDA7ud4fLxrZBWukI78JlfA64Nxz5QV/DaHFuwNqxoSWDOZzPszYeLyEzJQP5RMcPHg5DjC+nrpK4qaruRwRf/YLj1ZtH2qnb4/5f1mOutArLXFukDHiBhf7PcMP82BC7aKwkjaKE0y8YdeLNHSdfA8x13pjGX7qgykYn3Bt9dmWITY6af6F/aWw1VO6PsF1zHS08g68Z24f7lhZymQHAdM0ttH993lEYrRgKc4ro8VqG0W+PxN3/mvyiTa6nWDVct3L1Ev+Yno/zjTsXwHbKObHm/hd685zJutb3LqTCrswud9be/BI2puDC6taGu6IPk1sHbc7r2iq8KK6rlcTrx2S+58NdEe92S5IfmqmbQlt+IUhoA/WrnTTVjpvF7g2PDedWuhQ1lXPJaz29+bmV35wsZIgZ4mYNVyPj/2ggGvk6hCbDp700sXzhniMMeN4FBg91ARB62NmfX6Z4kvuJyvUclxfo2d6faS80h5OoEQ70ZkPMNu+pkwy147G47I1xJwuikcbmSGEufsXtQRAzzE2bIZWY9knncXK8ZbjzC/6yCrUQAeCz2Tg076x3JK9cqb5RuvOvqbe3JxhlwuTFZuD/uyNc2SlXhWHOsPbWCJJEGg3GM+F2R3vvd+uG1szu2QTJHlkC1e0mBcfAv74dyYJZLtrYIrNjNrVIDCKonzhj+dTg7SUzbRpPmsD19RiEokexni6ahRMPTTbOUi5OPOaXlnszWh6zGoglIpT7IDSOW7+b3GvmVnYkoQg7jrFZriwa3HQrm4l/srUk7X1xuq/xEwuc+BKZSrk3ERL+dKpeRIBSvxVH9ZKfuTXai0RI/bRtooWgXD2pJDYCQmHyv/0bv+iptfP/ABcqQ2kmqAAA")))
open("/content/montage/logo_locapay.png", "wb").write(base64.b64decode("..."))
sys.path.insert(0, "/content/montage")
import importlib, montage_auto as M; importlib.reload(M)
import torch
print("✅ Installation terminée —", "GPU actif 🚀" if torch.cuda.is_available() else "⚠️ pas de GPU (ça marchera, mais plus lentement)")

In [ ]:
#@title 2️⃣ Réglages
#@markdown **Couleurs des écrans de couleur et de l'écran de fin**
couleur_1 = "#2E2C51" #@param {type:"string"}
couleur_2 = "#0AC2A4" #@param {type:"string"}
ecrans_de_couleur = 2 #@param {type:"slider", min:0, max:4, step:1}
#@markdown **Sous-titres et découpe**
position_sous_titres = 0.66 #@param {type:"slider", min:0.40, max:0.85, step:0.01}
couper_les_hesitations = True #@param {type:"boolean"}
pause_max_conservee = 0.45 #@param {type:"slider", min:0.20, max:1.00, step:0.05}
#@markdown **Son**
effets_sonores = "léger" #@param ["léger", "aucun"]
ajouter_une_musique = False #@param {type:"boolean"}
volume_musique = 0.18 #@param {type:"slider", min:0.05, max:0.50, step:0.01}
#@markdown **Logo** (cochez « autre_logo » pour charger un PNG à la place du logo Locapay)
logo = True #@param {type:"boolean"}
autre_logo = False #@param {type:"boolean"}
#@markdown **Transcription** — mots rares à bien reconnaître, séparés par des virgules
vocabulaire = "Locapay, démarcheurs, loyer, propriétaire, locataire, Cotonou, Bénin" #@param {type:"string"}
modele = "turbo" #@param ["turbo", "large-v3", "medium", "small"]
#@markdown **Vidéo déjà sur Google Drive ?** Indiquez son chemin (ex. `MyDrive/Videos/rush.mp4`), sinon laissez vide.
video_sur_drive = "" #@param {type:"string"}

R = M.Reglages(couleur_1=couleur_1, couleur_2=couleur_2, nb_ecrans_couleur=ecrans_de_couleur,
               position_sous_titres=position_sous_titres, supprimer_hesitations=couper_les_hesitations,
               pause_max=pause_max_conservee, effets_sonores=effets_sonores, volume_musique=volume_musique,
               logo_en_coin=logo, fichier_logo="/content/montage/logo_locapay.png" if logo else "",
               police_serif="/content/montage/polices/serif.ttf", police_grasse="/content/montage/polices/gras.ttf")
print("✅ Réglages enregistrés")

In [ ]:
#@title 3️⃣ Charger la vidéo (et la musique)
from google.colab import files
import shutil
os.makedirs("/content/montage/entree", exist_ok=True)
EXT_V = (".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm", ".3gp")
if video_sur_drive.strip():
    from google.colab import drive
    drive.mount("/content/drive")
    VIDEO = os.path.join("/content/drive", video_sur_drive.strip().lstrip("/"))
    assert os.path.exists(VIDEO), f"Fichier introuvable : {VIDEO}"
else:
    print("👉 Sélectionnez la vidéo brute :")
    up = files.upload()
    vids = [n for n in up if n.lower().endswith(EXT_V)]
    assert vids, "Aucune vidéo reconnue (formats : mp4, mov, m4v, avi, mkv, webm)."
    VIDEO = f"/content/montage/entree/{vids[0]}"
    shutil.move(vids[0], VIDEO)
MUSIQUE = None
if ajouter_une_musique:
    print("👉 Sélectionnez la musique (mp3, wav, m4a…) :")
    up = files.upload()
    if up:
        n = list(up)[0]; MUSIQUE = f"/content/montage/entree/{n}"; shutil.move(n, MUSIQUE)
if logo and autre_logo:
    print("👉 Sélectionnez votre logo (PNG, fond transparent de préférence) :")
    up = files.upload()
    if up:
        n = list(up)[0]; R.fichier_logo = f"/content/montage/entree/{n}"; shutil.move(n, R.fichier_logo)
DUREE = M.duree_video(VIDEO)
print(f"✅ Vidéo : {os.path.basename(VIDEO)} ({DUREE:.1f} s)" + (f" — musique : {os.path.basename(MUSIQUE)}" if MUSIQUE else ""))

In [ ]:
#@title 4️⃣ Transcription automatique
mots_bruts = M.transcrire(VIDEO, modele=modele, langue="fr", vocabulaire=vocabulaire)
print("\n📝 Transcription :\n")
print(M.texte_transcription(mots_bruts))

In [ ]:
#@title 5️⃣ Corrections (facultatif) et plan de montage
#@markdown Format : `mot_mal_reconnu=bon_mot; autre=correction` — ex. `Locapé=Locapay; démarcheur=démarcheurs`
corrections = "Locapé=Locapay; Lokapay=Locapay; Locapaï=Locapay" #@param {type:"string"}
mots = M.appliquer_corrections(mots_bruts, corrections)
PLAN = M.construire_plan(mots, DUREE, R)
print(M.resume_plan(PLAN))
print("\nLes mots en MAJUSCULES seront en gras. Relancez cette étape après une correction.")

In [ ]:
#@title 6️⃣ Montage et téléchargement
from IPython.display import Video, display
os.makedirs("/content/montage/sortie", exist_ok=True)
base = os.path.splitext(os.path.basename(VIDEO))[0]
SORTIE = f"/content/montage/sortie/{base}_monte.mp4"
print("🎬 Montage en cours…")
M.monter(VIDEO, PLAN, R, SORTIE, musique=MUSIQUE)
M.ecrire_srt(PLAN, SORTIE.replace(".mp4", ".srt"))
open(SORTIE.replace(".mp4", "_transcription.txt"), "w", encoding="utf-8").write(M.texte_transcription(mots))
print("✅ Terminé !")
display(Video(SORTIE, embed=True, width=320))
files.download(SORTIE)

---
**Fichiers produits** (dossier `/content/montage/sortie`, panneau 📁 à gauche) : la vidéo montée, les sous-titres `.srt` et la transcription `.txt`.

**Astuces**
- Filmez en vertical, la personne au centre, avec un son propre : la transcription n'en sera que meilleure.
- Trop de coupes ? Montez « pause_max_conservee » (étape 2). Pas assez ? Baissez-la.
- Sous-titres qui cachent les mains ou le visage ? Ajustez « position_sous_titres ».